In [ ]:
# Run once in a fresh notebook environment.
%pip install -q numpy scipy pandas matplotlib

# Allocation Objectives and Solver Diagnostics

Reproduce full-information objective diagnosis, repeated-information value, independent active-search confirmation and paired non-myopic solver comparisons. Every intermediate is generated by the public CLI from built-in environments or scientific configurations. No cached result table or prepared private input is required.

The full recipe retains each stage's episode count, task size, seed namespace, allocation tolerance and solver settings. Discovery and confirmation are distinct; shared myopic/DP episodes support paired utility comparisons. Failed confirmation and mixed solver findings remain evidence, not targets to tune away.

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "pyproject.toml").exists() and (p / "scripts").is_dir())
RUN = False
MODE = "full"  # "smoke" is reduced-scale wiring, not report evidence.
assert MODE in ("smoke", "full")
OUTPUT = Path("results/allocation_mechanisms") / MODE
SMOKE_EPISODES = 5
VOI_SAMPLES = 500 if MODE == "full" else 24
ORACLE_GRID_SIZE = 4001
ALLOCATION_TOLERANCE = 0.05
SAMPLE_BUDGETS = "0,2,4,6,8,10,12"
DP_MAX_SAMPLES = 10
DP_MEAN_GRID_SIZE = 50
DP_MEAN_GRID_RADIUS_SD = 3.0
DP_OBSERVATION_BRANCHES = 7
DP_OBSERVATION_INTEGRATION = "gauss_hermite"

# family, episodes/environment, episodes/task, seed namespace offset
RECIPES = {
    "oracle": ("oracle", 1200, 100, 0),
    "discovery": ("six_sample", 120, 5, 0),
    "cost": ("custom_rr", 120, 5, 0),
    "oat": ("custom_rr", 120, 5, 0),
    "fixed_budget": ("fixed_budget", 1200, 5, 300000),
    "confirmation": ("custom_rr", 1200, 10, 1000000),
    "myopic": ("custom_rr", 1200, 10, 900000),
    "dp": ("custom_rr", 1200, 1200, 900000),
}
if RUN and (PROJECT_ROOT / OUTPUT).exists() and any((PROJECT_ROOT / OUTPUT).iterdir()):
    raise ValueError("Choose a fresh OUTPUT directory; do not mix independent executions.")
print({"run": RUN, "mode": MODE, "output": str(OUTPUT), "full_recipes": RECIPES})

## Shared CLI Orchestration

The helper freezes a manifest, executes every task, then strictly collects it. With `RUN = False`, it prints create/run-task/collect commands without reading missing generated files; task zero is shown as the representative runnable command. Actual execution reads the newly created manifest and traverses every task.

Smoke keeps the same public producer/reader chain, seed namespaces and frozen selection rules, but uses five episodes/environment, at most five/task and 24 VOI draws. A low-precision run may fail anchor or candidate selection; this remains a visible failure, never a reason to inject a fallback candidate. The report's full-count gate is not run in smoke.

In [ ]:
WORKFLOW = "scripts/active_search_evaluation_workflow.py"
ANALYZE = "scripts/analyze_active_search.py"

def execute(arguments, enabled=True):
    command = [sys.executable, *map(str, arguments)]
    print(shlex.join(command))
    if RUN and enabled:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    elif not enabled:
        print("Full-count report only: not executed in smoke.")

def evaluate(name, configs=None, policy="myopic_voi"):
    family, episodes, per_task, seed_offset = RECIPES[name]
    if MODE == "smoke":
        episodes, per_task = SMOKE_EPISODES, min(per_task, SMOKE_EPISODES)
    directory = OUTPUT / name
    manifest = directory / "active_search_manifest.json"
    arguments = [
        WORKFLOW, "create", "--family", family, "--output-dir", directory,
        "--episodes", episodes, "--episodes-per-task", per_task,
        "--observation-draws", VOI_SAMPLES, "--allocation-tolerance", ALLOCATION_TOLERANCE,
        "--oracle-grid-size", ORACLE_GRID_SIZE, "--seed-namespace-offset", seed_offset,
        "--sample-budgets", SAMPLE_BUDGETS, "--rr-policy", policy,
        "--dp-max-samples", DP_MAX_SAMPLES, "--dp-mean-grid-size", DP_MEAN_GRID_SIZE,
        "--dp-mean-grid-radius-sd", DP_MEAN_GRID_RADIUS_SD,
        "--dp-observation-branches", DP_OBSERVATION_BRANCHES,
        "--dp-observation-integration", DP_OBSERVATION_INTEGRATION,
    ]
    if configs is not None:
        arguments.extend(["--configs-json", configs])
    execute(arguments)
    if RUN:
        frozen = json.loads((PROJECT_ROOT / manifest).read_text())
        for task in frozen["tasks"]:
            execute([WORKFLOW, "run-task", "--manifest", manifest,
                     "--task-index", task["task_index"]])
    else:
        execute([WORKFLOW, "run-task", "--manifest", manifest, "--task-index", 0])
        print("Execution will repeat run-task for every index in the newly generated manifest.")
    execute([WORKFLOW, "collect", "--manifest", manifest])
    return directory

## Objective and Repeated-Information Value

Oracle episodes produce sign-stratified summaries, four selected anchors, sampling-cost configurations and one-factor-at-a-time configurations. The generated anchors also drive the fixed-budget comparison. All selected configurations are recomputed, not supplied as stored results.

In [ ]:
oracle = evaluate("oracle")
objective = OUTPUT / "objective"
execute([ANALYZE, "oracle",
         "--episodes", oracle / "active_search_oracle_episodes.csv",
         "--manifest", oracle / "active_search_manifest.json",
         "--anchor-count", 4, "--output-dir", objective])
cost = evaluate("cost", objective / "active_search_sampling_cost_sweep_configs.json")
oat = evaluate("oat", objective / "active_search_oat_sweep_configs.json")
fixed = evaluate("fixed_budget", objective / "active_search_oracle_anchors.json")
formal = OUTPUT / "formal"
execute([ANALYZE, "formal-summaries",
         "--sampling-summary", cost / "active_search_rr_environment_summary.csv",
         "--sampling-manifest", cost / "active_search_manifest.json",
         "--oat-summary", oat / "active_search_rr_environment_summary.csv",
         "--oat-manifest", oat / "active_search_manifest.json",
         "--fixed-budget-summary", fixed / "active_search_fixed_budget_summary.csv",
         "--output-dir", formal])

## Discovery and Independent Confirmation

Discovery uses 120 episodes/environment with seed offset zero. The shared selection rule writes at most 12 frozen configurations before confirmation, which uses 1,200 episodes/environment and seed offset 1,000,000. Joint point estimates and the stricter two-lower-bound confirmation are separate outcomes; retain both.

In [ ]:
discovery = evaluate("discovery")
confirmation_configs = OUTPUT / "confirmation_configs.json"
execute([ANALYZE, "select-confirmation",
         "--summary", discovery / "active_search_rr_environment_summary.csv",
         "--manifest", discovery / "active_search_manifest.json",
         "--max-candidates", 12, "--output", confirmation_configs])
confirmation = evaluate("confirmation", confirmation_configs)

## Paired Held-Out Solver Comparison

Both policies use the three environments in `configs/solver_heldout_configs.json`, 1,200 episodes/environment and seed offset 900,000. DP freezes 10 maximum samples, a 50-point mean grid spanning 3 SD and seven Gauss-Hermite observation branches. DP uses one 1,200-episode task per environment; myopic uses ten-episode tasks. The shared analyzer checks episode identities and common observations before computing paired utility differences.

In [ ]:
heldout = Path("configs/solver_heldout_configs.json")
myopic = evaluate("myopic", heldout, "myopic_voi")
dp = evaluate("dp", heldout, "discretized_dp")
solver = OUTPUT / "solver"
execute([ANALYZE, "solver-comparison",
         "--myopic-episodes", myopic / "active_search_rr_episodes.csv",
         "--myopic-manifest", myopic / "active_search_manifest.json",
         "--dp-episodes", dp / "active_search_rr_episodes.csv",
         "--dp-manifest", dp / "active_search_manifest.json",
         "--output-dir", solver])

## Validated Report and Output Readers

Full mode regenerates all six report inputs above before generating the report, supporting tables and figures. The report requires oracle 129,600 episodes / 1,296 tasks / 108 summaries, discovery 38,880 / 7,776 / 324 and confirmation 14,400 / 1,440 / 12. Its other scientific consistency checks are not bypassed. Smoke only previews this command and is not labelled report reproduction.

In [ ]:
REPORT_INPUTS = {
    "oracle-dir": oracle,
    "oracle-analysis-dir": objective,
    "formal-dir": formal,
    "discovery-dir": discovery,
    "confirmation-dir": confirmation,
    "solver-dir": solver,
    "output-dir": OUTPUT / "report",
}
report_arguments = ["scripts/generate_active_search_report.py"]
for key, value in REPORT_INPUTS.items():
    report_arguments.extend(["--" + key, value])
execute(report_arguments, enabled=MODE == "full")

In [ ]:
import pandas as pd

if RUN:
    for directory, name in (
        (objective, "active_search_oracle_analysis.json"),
        (formal, "active_search_formal_summary.json"),
        (solver, "active_search_solver_comparison_summary.json"),
    ):
        display(json.loads((PROJECT_ROOT / directory / name).read_text()))
    for directory, name in (
        (objective, "active_search_oracle_sign_summary.csv"),
        (formal, "active_search_formal_sweep_long.csv"),
        (formal, "active_search_fixed_budget_evidence.csv"),
        (confirmation, "active_search_rr_environment_summary.csv"),
        (solver, "active_search_solver_comparison.csv"),
    ):
        display(pd.read_csv(PROJECT_ROOT / directory / name))
    if MODE == "full":
        supporting = PROJECT_ROOT / REPORT_INPUTS["output-dir"] / "supporting_data"
        display(json.loads((supporting / "active_search_report_summary.json").read_text()))
        display(pd.read_csv(supporting / "active_search_confirmation_comparison.csv"))

## Interpretation

Expected average utility is the performance criterion. Allocation resemblance, true-outcome equality and sample counts diagnose behavior; they do not establish identical cognitive rules. These Gaussian objective/solver analyses are distinct from the capped finite tutoring task. A numerical approximation is not an exact continuous-state metalevel optimum. Do not tune seeds, thresholds or selected environments to force the reported conclusion.